# 🌿 Plant Disease Classification using Deep Learning
## Transfer Learning with EfficientNetB3 on PlantVillage Dataset

---
**Dataset:** [PlantVillage Dataset](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset)  
**Model:** EfficientNetB3 (Transfer Learning + Fine-Tuning)  
**Framework:** TensorFlow / Keras  
**Classes:** 38 plant disease / healthy categories  

### Pipeline Overview
1. 📂 Dataset Loading & Exploration
2. 🔄 Data Preprocessing & Augmentation
3. 🏗️ Model Building (Transfer Learning)
4. 🎯 Model Training (Frozen → Fine-tune)
5. 📊 Model Evaluation
6. 🔥 Grad-CAM Visualization
7. 🔮 Prediction on New Image
8. 💾 Save the Model
---

## ⚙️ Environment Setup
Install and import all required libraries.

In [ ]:
# Install any missing packages (run once)
!pip install -q tensorflow matplotlib seaborn scikit-learn numpy pillow opencv-python-headless

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image
from collections import defaultdict

# ── TensorFlow / Keras ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                         ReduceLROnPlateau, TensorBoard)
from tensorflow.keras.optimizers import Adam

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

---
## 📂 Section 1 — Dataset Loading & Exploration

We load the PlantVillage dataset from Kaggle, enumerate all class folders, count images per class, and display sample leaves from different disease categories.

In [ ]:
# ── Dataset path configuration ───────────────────────────────────────────────
# The dataset has three sub-folders: color, grayscale, segmented
# We will use the 'color' variant for the best visual features
DATASET_BASE = '/kaggle/input/plantvillage-dataset/plantvillage dataset/color'

# Verify the path exists
if not os.path.exists(DATASET_BASE):
    # Fallback: try alternate path structures present in different Kaggle versions
    alt_paths = [
        '/kaggle/input/plantvillage-dataset/color',
        '/kaggle/input/plantvillage-dataset/PlantVillage',
    ]
    for p in alt_paths:
        if os.path.exists(p):
            DATASET_BASE = p
            break

print(f'Dataset path  : {DATASET_BASE}')
print(f'Path exists   : {os.path.exists(DATASET_BASE)}')

# List top-level contents to confirm
top = os.listdir(DATASET_BASE)[:5]
print(f'Sample folders: {top}')

In [ ]:
# ── Enumerate classes and count images ───────────────────────────────────────
class_counts = {}
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp'}

for class_name in sorted(os.listdir(DATASET_BASE)):
    class_dir = os.path.join(DATASET_BASE, class_name)
    if os.path.isdir(class_dir):
        count = sum(1 for f in os.listdir(class_dir)
                    if Path(f).suffix.lower() in VALID_EXTS)
        class_counts[class_name] = count

CLASS_NAMES  = list(class_counts.keys())
NUM_CLASSES  = len(CLASS_NAMES)
TOTAL_IMAGES = sum(class_counts.values())

print(f'Total classes : {NUM_CLASSES}')
print(f'Total images  : {TOTAL_IMAGES:,}')
print()
print('Class distribution (images per class):')
for i, (cls, cnt) in enumerate(class_counts.items(), 1):
    print(f'  {i:2}. {cls:<55} {cnt:5,}')

In [ ]:
# ── Plot class distribution bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 6))

colors = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
bars   = ax.bar(range(NUM_CLASSES), list(class_counts.values()), color=colors)

ax.set_title('Class Distribution — PlantVillage Dataset', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Disease Class Index', fontsize=12)
ax.set_ylabel('Number of Images',    fontsize=12)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(range(NUM_CLASSES), fontsize=8)
ax.axhline(TOTAL_IMAGES / NUM_CLASSES, color='red', linestyle='--',
            linewidth=1.5, label=f'Mean = {TOTAL_IMAGES//NUM_CLASSES:,}')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → class_distribution.png')

In [ ]:
# ── Display sample images from each class (first 20 classes) ─────────────────
SHOW_N = min(20, NUM_CLASSES)
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()

for idx, cls in enumerate(CLASS_NAMES[:SHOW_N]):
    cls_dir = os.path.join(DATASET_BASE, cls)
    imgs    = [f for f in os.listdir(cls_dir) if Path(f).suffix.lower() in VALID_EXTS]
    if not imgs:
        continue
    img_path = os.path.join(cls_dir, random.choice(imgs))
    img      = Image.open(img_path).convert('RGB')
    axes[idx].imshow(img)
    # Shorten label to fit
    short = cls.replace('___', '\n').replace('_', ' ')
    axes[idx].set_title(short, fontsize=7, pad=4)
    axes[idx].axis('off')

for ax in axes[SHOW_N:]:
    ax.axis('off')

plt.suptitle('Sample Images — PlantVillage Dataset (First 20 Classes)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → sample_images.png')

---
## 🔄 Section 2 — Data Preprocessing & Augmentation

**Strategy:**
- Resize all images to **224 × 224** (EfficientNetB3 default)
- Normalize pixel values to **[0, 1]**
- Apply aggressive **data augmentation** only on the training set
- Split: **70% train | 20% validation | 10% test** using `ImageDataGenerator.flow_from_directory`

We first build a CSV dataframe of all image paths + labels, split it, then feed each split to its own generator.

In [ ]:
# ── Build image-path / label dataframe ───────────────────────────────────────
records = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_BASE, cls)
    for fname in os.listdir(cls_dir):
        if Path(fname).suffix.lower() in VALID_EXTS:
            records.append({'filepath': os.path.join(cls_dir, fname), 'label': cls})

df = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Total records : {len(df):,}')
print(df.head())

In [ ]:
# ── Stratified split: 70 / 20 / 10 ───────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# First split: 90% train+val, 10% test
df_trainval, df_test = train_test_split(
    df, test_size=0.10, stratify=df['label'], random_state=SEED)

# Second split: 70% train, 20% val  (0.222 of 90% ≈ 20% of total)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.222, stratify=df_trainval['label'], random_state=SEED)

print(f'Train   : {len(df_train):6,}  ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val     : {len(df_val):6,}  ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test    : {len(df_test):6,}  ({len(df_test)/len(df)*100:.1f}%)')

In [ ]:
# ── ImageDataGenerators ───────────────────────────────────────────────────────

# Training generator — with augmentation
train_gen = ImageDataGenerator(
    rescale          = 1./255,
    rotation_range   = 30,
    width_shift_range= 0.2,
    height_shift_range=0.2,
    zoom_range       = 0.2,
    horizontal_flip  = True,
    vertical_flip    = True,
    brightness_range = [0.8, 1.2],
    fill_mode        = 'nearest'
)

# Validation & test generators — only rescale (no augmentation)
val_test_gen = ImageDataGenerator(rescale=1./255)


def make_generator(gen, dataframe, shuffle=True):
    """Create a flow_from_dataframe generator."""
    return gen.flow_from_dataframe(
        dataframe       = dataframe,
        x_col           = 'filepath',
        y_col           = 'label',
        target_size     = IMG_SIZE,
        batch_size      = BATCH_SIZE,
        class_mode      = 'categorical',
        shuffle         = shuffle,
        seed            = SEED
    )


train_ds = make_generator(train_gen,    df_train, shuffle=True)
val_ds   = make_generator(val_test_gen, df_val,   shuffle=False)
test_ds  = make_generator(val_test_gen, df_test,  shuffle=False)

# Retrieve class-index mapping (needed for evaluation)
CLASS_INDICES = train_ds.class_indices               # {'Apple___Apple_scab': 0, ...}
IDX_TO_CLASS  = {v: k for k, v in CLASS_INDICES.items()}

print(f'Classes in generator : {train_ds.num_classes}')
print(f'Training steps/epoch : {len(train_ds)}')

In [ ]:
# ── Visualise augmented batch ─────────────────────────────────────────────────
sample_batch_x, sample_batch_y = next(train_ds)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()
for i in range(18):
    axes[i].imshow(sample_batch_x[i])
    lbl = IDX_TO_CLASS[np.argmax(sample_batch_y[i])].replace('___', '\n').replace('_', ' ')
    axes[i].set_title(lbl, fontsize=6)
    axes[i].axis('off')

plt.suptitle('Sample Augmented Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏗️ Section 3 — Model Building (Transfer Learning)

We use **EfficientNetB3** pretrained on ImageNet as the backbone.

**Architecture:**
```
EfficientNetB3 (frozen)  →  GlobalAveragePooling2D  →  BatchNorm  →  Dropout(0.3)  →  Dense(38, softmax)
```

The base layers are **frozen** first so we only train the new head, then we **fine-tune** top layers in Section 4.

In [ ]:
# ── Build the transfer-learning model ─────────────────────────────────────────
def build_model(num_classes: int, img_size=(224, 224), trainable_base=False):
    """
    Build EfficientNetB3-based classifier.

    Parameters
    ----------
    num_classes    : number of output classes
    img_size       : (height, width) tuple
    trainable_base : whether to unfreeze the base model
    """
    inputs = keras.Input(shape=(*img_size, 3), name='input_layer')

    # ── Base model (EfficientNetB3 expects [0,255] — we handle rescaling above) ──
    # NOTE: include_preprocessing=False because our generator already rescales to [0,1].
    # EfficientNet's internal normalisation divides by 255 again, so we must
    # scale our [0,1] images **back** to [0,255] before passing them in:
    x = layers.Rescaling(255.0, name='rescale_to_255')(inputs)   # [0,1] → [0,255]

    base_model = EfficientNetB3(
        include_top  = False,
        weights      = 'imagenet',
        input_tensor = x,
        pooling      = None
    )
    base_model.trainable = trainable_base

    # ── Custom head ──────────────────────────────────────────────────────────
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=base_model.input, outputs=outputs, name='PlantDiseaseClassifier')
    return model, base_model


model, base_model = build_model(NUM_CLASSES, IMG_SIZE, trainable_base=False)

model.compile(
    optimizer = Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

print(f'Total params         : {model.count_params():,}')
print(f'Trainable params     : {sum([np.prod(v.shape) for v in model.trainable_variables]):,}')
print(f'Non-trainable params : {sum([np.prod(v.shape) for v in model.non_trainable_variables]):,}')

---
## 🎯 Section 4 — Model Training

**Phase 1 — Head training (base frozen):** Train only the custom head for `PHASE1_EPOCHS` epochs.  
**Phase 2 — Fine-tuning (top layers unfrozen):** Unfreeze the last `UNFREEZE_LAYERS` layers of EfficientNetB3 and train at a smaller learning rate.

**Callbacks:**
- `EarlyStopping` — stops when val_accuracy plateaus
- `ModelCheckpoint` — saves the best weights automatically
- `ReduceLROnPlateau` — slashes the LR when training stalls

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
PHASE1_EPOCHS  = 12   # frozen-base training
PHASE2_EPOCHS  = 8    # fine-tuning
UNFREEZE_LAYERS = 30  # how many layers from top to unfreeze during fine-tune

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath='best_model_phase1.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath='best_model_phase2.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1)
]

print('Callbacks configured ✓')

In [ ]:
# ── Phase 1: Train head only ──────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1 — Training classification head (base frozen)')
print('=' * 60)

history1 = model.fit(
    train_ds,
    epochs           = PHASE1_EPOCHS,
    validation_data  = val_ds,
    callbacks        = callbacks_phase1,
    verbose          = 1
)

In [ ]:
# ── Phase 2: Fine-tune top layers ─────────────────────────────────────────────
print('=' * 60)
print(f'PHASE 2 — Fine-tuning top {UNFREEZE_LAYERS} layers of EfficientNetB3')
print('=' * 60)

# Unfreeze the last UNFREEZE_LAYERS layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# Re-compile with a lower learning rate for fine-tuning
model.compile(
    optimizer = Adam(learning_rate=1e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

trainable_now = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'Trainable params (phase 2): {trainable_now:,}')

history2 = model.fit(
    train_ds,
    epochs           = PHASE2_EPOCHS,
    validation_data  = val_ds,
    callbacks        = callbacks_phase2,
    verbose          = 1
)

In [ ]:
# ── Plot training & validation curves ─────────────────────────────────────────
def merge_histories(h1, h2):
    """Concatenate two Keras history objects."""
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history.get(k, [])
    return merged

history = merge_histories(history1, history2)
epochs  = range(1, len(history['accuracy']) + 1)
phase1_end = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Accuracy plot ---
axes[0].plot(epochs, history['accuracy'],     'b-o', ms=4, label='Train Acc')
axes[0].plot(epochs, history['val_accuracy'], 'r-o', ms=4, label='Val Acc')
axes[0].axvline(phase1_end, color='gray', linestyle='--', label=f'Fine-tune starts (ep {phase1_end+1})')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- Loss plot ---
axes[1].plot(epochs, history['loss'],     'b-o', ms=4, label='Train Loss')
axes[1].plot(epochs, history['val_loss'], 'r-o', ms=4, label='Val Loss')
axes[1].axvline(phase1_end, color='gray', linestyle='--', label=f'Fine-tune starts (ep {phase1_end+1})')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training History — Phase 1 (Frozen) + Phase 2 (Fine-tune)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📊 Section 5 — Model Evaluation

We evaluate on the **held-out test set** and produce:
1. Overall accuracy & top-3 accuracy
2. Precision / Recall / F1 per class (Classification Report)
3. Confusion Matrix heatmap

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
test_loss, test_acc, test_top3 = model.evaluate(test_ds, verbose=1)
print(f'\nTest  Loss       : {test_loss:.4f}')
print(f'Test  Accuracy   : {test_acc*100:.2f}%')
print(f'Test  Top-3 Acc  : {test_top3*100:.2f}%')

In [ ]:
# ── Generate predictions for Classification Report & Confusion Matrix ─────────
print('Generating predictions on test set …')

test_ds.reset()
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_ds.classes           # true integer labels

label_names  = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
report = classification_report(y_true, y_pred, target_names=label_names, digits=3)
print(report)

# Save report to file
with open('classification_report.txt', 'w') as f:
    f.write(report)
print('Saved → classification_report.txt')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names,
    linewidths=0.3, linecolor='lightgray',
    annot_kws={'size': 7}
)
ax.set_title('Confusion Matrix — Test Set', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → confusion_matrix.png')

---
## 🔥 Section 6 — Grad-CAM Visualization

**Gradient-weighted Class Activation Mapping (Grad-CAM)** highlights the image regions that influenced the model's prediction most strongly.

**Algorithm:**
1. Forward-pass a sample image
2. Compute the gradient of the predicted class score w.r.t. the last convolutional layer's feature maps
3. Pool gradients spatially → importance weights
4. Take the weighted sum of feature maps → heatmap
5. Overlay heatmap on the original image

In [ ]:
# ── Grad-CAM implementation ───────────────────────────────────────────────────
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    """
    Compute Grad-CAM heatmap.

    Parameters
    ----------
    model               : trained Keras model
    img_array           : preprocessed image [1, H, W, 3]
    last_conv_layer_name: name of the last conv layer to visualise
    pred_index          : class index to explain (None = argmax of output)

    Returns
    -------
    heatmap : np.ndarray, shape (h, w), values in [0, 1]
    """
    # Create a model that outputs (conv feature maps, final predictions)
    grad_model = Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradient of the class score w.r.t. the conv feature map
    grads = tape.gradient(class_channel, conv_outputs)

    # Global average pool the gradients → channel-wise importance weights
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight each feature map channel by its importance
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize to [0, 1]; clip negatives (only positive activations matter)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(original_img, heatmap, alpha=0.4):
    """
    Superimpose the Grad-CAM heatmap on the original image.

    Parameters
    ----------
    original_img : np.ndarray uint8 [H, W, 3]
    heatmap      : np.ndarray float [h, w]
    alpha        : transparency of heatmap overlay

    Returns
    -------
    superimposed : np.ndarray uint8 [H, W, 3]
    """
    # Resize heatmap to match the image
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    # Apply colormap
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    # Blend
    superimposed = cv2.addWeighted(original_img, 1 - alpha, heatmap_colored, alpha, 0)
    return superimposed


# Name of EfficientNetB3's last convolutional layer
LAST_CONV_LAYER = 'top_conv'

# Verify the layer exists
layer_names = [l.name for l in model.layers]
if LAST_CONV_LAYER not in layer_names:
    # Fallback: find last Conv2D layer automatically
    for l in reversed(model.layers):
        if isinstance(l, layers.Conv2D):
            LAST_CONV_LAYER = l.name
            break
print(f'Using Grad-CAM layer: {LAST_CONV_LAYER}')

In [ ]:
# ── Run Grad-CAM on 5 sample test images ─────────────────────────────────────
NUM_SAMPLES = 5

# Pick 5 random images from the test dataframe
sample_rows = df_test.sample(n=NUM_SAMPLES, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(15, NUM_SAMPLES * 4))

for i, row in sample_rows.iterrows():
    img_path  = row['filepath']
    true_label = row['label']

    # ── Load & preprocess ──
    orig_img  = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))  # uint8
    proc_img  = orig_img.astype(np.float32) / 255.0
    inp       = np.expand_dims(proc_img, axis=0)   # (1, 224, 224, 3)

    # ── Predict ──
    preds     = model.predict(inp, verbose=0)[0]
    pred_idx  = np.argmax(preds)
    pred_class= IDX_TO_CLASS[pred_idx]
    confidence= preds[pred_idx]

    # ── Grad-CAM ──
    heatmap   = get_gradcam_heatmap(model, inp, LAST_CONV_LAYER, pred_index=pred_idx)
    overlay   = overlay_gradcam(orig_img, heatmap, alpha=0.45)

    # ── Plot ──
    axes[i][0].imshow(orig_img)
    axes[i][0].set_title(f'Original\nTrue: {true_label.replace("___", " | ").replace("_", " ")}',
                         fontsize=8)
    axes[i][0].axis('off')

    axes[i][1].imshow(heatmap, cmap='jet')
    axes[i][1].set_title('Grad-CAM Heatmap', fontsize=8)
    axes[i][1].axis('off')

    color = 'green' if pred_class == true_label else 'red'
    axes[i][2].imshow(overlay)
    axes[i][2].set_title(
        f'Overlay\nPred: {pred_class.replace("___", " | ").replace("_", " ")}\n'
        f'Conf: {confidence*100:.1f}%',
        fontsize=8, color=color)
    axes[i][2].axis('off')

plt.suptitle('Grad-CAM Visualization — Model Attention on Leaf', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('gradcam_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → gradcam_visualization.png')

---
## 🔮 Section 7 — Prediction on New Image

A clean, reusable **`predict_disease(image_path)`** function that:
1. Loads and preprocesses the image
2. Runs inference
3. Returns the class name + confidence (and top-5 predictions).

We then demonstrate it on a sample image from the dataset.

In [ ]:
# ── Prediction function ───────────────────────────────────────────────────────
def predict_disease(image_path: str,
                    model      = model,
                    idx_to_cls = IDX_TO_CLASS,
                    img_size   = IMG_SIZE,
                    top_k      = 5) -> dict:
    """
    Predict the plant disease from an image file.

    Parameters
    ----------
    image_path  : str  – path to the input image
    model       : trained Keras model
    idx_to_cls  : dict  – mapping from integer index to class name
    img_size    : (int, int) – target size used during training
    top_k       : int  – number of top predictions to return

    Returns
    -------
    dict with keys:
        'class'      : str   – predicted class name
        'confidence' : float – confidence in [0, 1]
        'top_k'      : list  – [(class, confidence), ...]
    """
    # 1. Load & preprocess
    img = Image.open(image_path).convert('RGB').resize(img_size)
    arr = np.array(img, dtype=np.float32) / 255.0          # normalise to [0,1]
    inp = np.expand_dims(arr, axis=0)                       # add batch dim

    # 2. Inference
    probs     = model.predict(inp, verbose=0)[0]

    # 3. Top-K results
    top_k_idx = np.argsort(probs)[::-1][:top_k]
    top_k_res = [(idx_to_cls[i], float(probs[i])) for i in top_k_idx]

    best_idx   = top_k_idx[0]
    best_cls   = idx_to_cls[best_idx]
    best_conf  = float(probs[best_idx])

    return {
        'class'      : best_cls,
        'confidence' : best_conf,
        'top_k'      : top_k_res
    }


# ── Visualise prediction on a sample image ────────────────────────────────────
sample_img_path = df_test.sample(1, random_state=99).iloc[0]['filepath']
true_lbl        = df_test[df_test['filepath'] == sample_img_path].iloc[0]['label']

result = predict_disease(sample_img_path)

print('=' * 55)
print(f' True label  : {true_lbl}')
print(f' Prediction  : {result["class"]}')
print(f' Confidence  : {result["confidence"]*100:.2f}%')
print(f' Correct?    : {result["class"] == true_lbl}')
print('─' * 55)
print(' Top-5 Predictions:')
for rank, (cls, conf) in enumerate(result['top_k'], 1):
    print(f'  {rank}. {cls:<50} {conf*100:.2f}%')
print('=' * 55)

In [ ]:
# ── Visual display of prediction ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Original image
img_show = Image.open(sample_img_path).convert('RGB')
axes[0].imshow(img_show)
axes[0].set_title(f'Input Image\nTrue: {true_lbl.replace("___", " | ").replace("_", " ")}',
                  fontsize=10)
axes[0].axis('off')

# Top-5 bar chart
classes = [c.replace('___', ' | ').replace('_', ' ') for c, _ in result['top_k']]
confs   = [conf * 100 for _, conf in result['top_k']]
colors  = ['#2ecc71' if c == result['class'].replace('___', ' | ').replace('_', ' ')
            else '#3498db' for c in classes]

bars = axes[1].barh(classes[::-1], confs[::-1], color=colors[::-1])
axes[1].set_title('Top-5 Predicted Classes', fontsize=10)
axes[1].set_xlabel('Confidence (%)')
axes[1].set_xlim(0, 105)
for bar, conf in zip(bars, confs[::-1]):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{conf:.1f}%', va='center', fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('predict_disease() — Demonstration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Section 8 — Save the Model

We save the model in **two formats**:
1. **`.h5` (HDF5)** — widely supported legacy format, easy to share
2. **TensorFlow SavedModel** — production format, supports TensorFlow Serving

In [ ]:
# ── Save in HDF5 format ───────────────────────────────────────────────────────
H5_PATH = 'plant_disease_model.h5'
model.save(H5_PATH)
size_mb = os.path.getsize(H5_PATH) / (1024 ** 2)
print(f'Model saved → {H5_PATH}  ({size_mb:.1f} MB)')

# ── Save in TensorFlow SavedModel format ─────────────────────────────────────
TF_SAVE_PATH = 'plant_disease_savedmodel'
model.save(TF_SAVE_PATH)
print(f'Model saved → {TF_SAVE_PATH}/')

# ── Save class index mapping for later inference ──────────────────────────────
import json
with open('class_indices.json', 'w') as f:
    json.dump(CLASS_INDICES, f, indent=2)
print('Class mapping saved → class_indices.json')

In [ ]:
# ── Verify the saved model loads correctly ────────────────────────────────────
print('Verifying saved model ...')
loaded_model = keras.models.load_model(H5_PATH)

# Quick sanity check: run the same sample prediction
img  = Image.open(sample_img_path).convert('RGB').resize(IMG_SIZE)
arr  = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)
p1   = model.predict(arr, verbose=0)[0]
p2   = loaded_model.predict(arr, verbose=0)[0]

assert np.allclose(p1, p2, atol=1e-5), 'Mismatch between original and loaded model!'
print('✅ Loaded model output matches original model — save successful!')
print(f'   Predicted class: {IDX_TO_CLASS[np.argmax(p2)]}  ({np.max(p2)*100:.2f}%)')

---
## ✅ Project Summary

| Stage                   | Details                                                   |
|-------------------------|-----------------------------------------------------------|
| **Dataset**             | PlantVillage — 38 classes, ~54K images                   |
| **Input size**          | 224 × 224 × 3 (RGB)                                       |
| **Base model**          | EfficientNetB3 pretrained on ImageNet                     |
| **Head**                | GAP → BatchNorm → Dropout(0.3) → Dense(38, softmax)       |
| **Training strategy**   | Phase 1: frozen base / Phase 2: fine-tune top 30 layers   |
| **Augmentation**        | Rotation, flip, zoom, shift, brightness                   |
| **Callbacks**           | EarlyStopping, ModelCheckpoint, ReduceLROnPlateau         |
| **Evaluation**          | Classification report + Confusion matrix on test split    |
| **Explainability**      | Grad-CAM heatmaps for 5 sample predictions                |
| **Saved formats**       | `.h5` + TensorFlow SavedModel                             |
| **Output files**        | `plant_disease_model.h5`, `class_indices.json`, PNGs      |

---
*Built with TensorFlow/Keras — EfficientNetB3 Transfer Learning*  
*PlantVillage Dataset — Kaggle: abdallahalidev/plantvillage-dataset*